In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install wfdb neurokit2 numpy scipy scikit-learn xgboost torch tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 708.4/708.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os, random, math
import numpy as np
from tqdm import tqdm

import wfdb
import neurokit2 as nk
from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---------------- CONFIG ----------------
SEED = 42
N_WINDOW = 30        # input RR window
H_HORIZON = 5        # predict next 5 RR values
RR_MIN, RR_MAX = 0.25, 2.50

# ECG preprocessing
BP_LOW, BP_HIGH = 0.5, 40.0
BP_ORDER = 3

# split (record-wise)
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# training
BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-5


torch: 2.6.0+cu124
DEVICE: cuda


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def bandpass_filter(ecg, fs, low=0.5, high=40.0, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, ecg).astype(np.float32)

def build_windows(rr, N, H):
    X, Y = [], []
    L = len(rr)
    for t in range(N - 1, L - H):
        x = rr[t-N+1:t+1]
        y = rr[t+1:t+1+H]
        if len(x) == N and len(y) == H:
            X.append(x)
            Y.append(y)
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def horizon_metrics(y_true, y_pred):
    H = y_true.shape[1]
    rmses, maes = [], []
    for k in range(H):
        rmses.append(rmse(y_true[:, k], y_pred[:, k]))
        maes.append(mae(y_true[:, k], y_pred[:, k]))
    return {
        "rmse_k": rmses,
        "mae_k": maes,
        "rmse_all": rmse(y_true.reshape(-1), y_pred.reshape(-1)),
        "mae_all": mae(y_true.reshape(-1), y_pred.reshape(-1)),
    }


In [5]:
DATA_DIR = "/kaggle/input/final-mith-dataset/mit-bih-polysomnographic-database-1.0.0"
# If your dataset folder name is different, open /kaggle/input and check:
!ls /kaggle/input


final-mith-dataset


In [6]:
print("Has RECORDS:", os.path.exists(os.path.join(DATA_DIR, "RECORDS")))


Has RECORDS: True


In [7]:
def read_records_list(data_dir):
    with open(os.path.join(data_dir, "RECORDS"), "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def pick_ecg_channel(sig_names):
    for i, name in enumerate(sig_names):
        if "ecg" in name.lower():
            return i
    return 0

def extract_rr_from_record(data_dir, record_name):
    try:
        rec = wfdb.rdrecord(os.path.join(data_dir, record_name))
        fs = float(rec.fs)
        sig = rec.p_signal
        if sig is None:
            return None

        ch_idx = pick_ecg_channel(rec.sig_name)
        ecg = sig[:, ch_idx].astype(np.float32)

        ecg_f = bandpass_filter(ecg, fs, BP_LOW, BP_HIGH, BP_ORDER)

        # R-peaks (NeuroKit2)
        _, info = nk.ecg_peaks(ecg_f, sampling_rate=fs)
        rpeaks = info.get("ECG_R_Peaks", None)
        if rpeaks is None or len(rpeaks) < (N_WINDOW + H_HORIZON + 10):
            return None
        rpeaks = np.asarray(rpeaks, dtype=np.int64)

        rr = np.diff(rpeaks) / fs
        rr = rr.astype(np.float32)

        mask = (rr >= RR_MIN) & (rr <= RR_MAX)
        rr = rr[mask]
        if rr.size < (N_WINDOW + H_HORIZON + 10):
            return None
        return rr
    except Exception:
        return None

records = read_records_list(DATA_DIR)
print("Total records:", len(records))
print("First 10:", records[:10])


Total records: 18
First 10: ['slp01a', 'slp01b', 'slp02a', 'slp02b', 'slp03', 'slp04', 'slp14', 'slp16', 'slp32', 'slp37']


In [8]:
X_all, Y_all = [], []
rec_ids_for_samples = []

for rec_name in tqdm(records):
    rr = extract_rr_from_record(DATA_DIR, rec_name)
    if rr is None:
        continue
    Xr, Yr = build_windows(rr, N_WINDOW, H_HORIZON)
    if Xr.size == 0:
        continue
    X_all.append(Xr)
    Y_all.append(Yr)
    rec_ids_for_samples.extend([rec_name] * Xr.shape[0])

X_all = np.concatenate(X_all, axis=0)
Y_all = np.concatenate(Y_all, axis=0)
rec_ids_for_samples = np.array(rec_ids_for_samples)

print("Samples:", X_all.shape[0])
print("X:", X_all.shape, "Y:", Y_all.shape)
print("Records used:", len(np.unique(rec_ids_for_samples)))


100%|██████████| 18/18 [00:37<00:00,  2.07s/it]

Samples: 367986
X: (367986, 30) Y: (367986, 5)
Records used: 18


In [9]:
unique_recs = np.unique(rec_ids_for_samples)

train_recs, test_recs = train_test_split(unique_recs, test_size=TEST_SIZE, random_state=SEED)
val_frac_of_train = VAL_SIZE / (1.0 - TEST_SIZE)
train_recs, val_recs = train_test_split(train_recs, test_size=val_frac_of_train, random_state=SEED)

def mask_for(recs):
    return np.isin(rec_ids_for_samples, recs)

m_train = mask_for(train_recs)
m_val = mask_for(val_recs)
m_test = mask_for(test_recs)

X_train, Y_train = X_all[m_train], Y_all[m_train]
X_val, Y_val = X_all[m_val], Y_all[m_val]
X_test, Y_test = X_all[m_test], Y_all[m_test]

print("Split samples:", len(X_train), len(X_val), len(X_test))

# scale using train RR only
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, 1))

def scale_seq(X):
    return scaler.transform(X.reshape(-1, 1)).reshape(X.shape).astype(np.float32)

X_train_s = scale_seq(X_train)
X_val_s = scale_seq(X_val)
X_test_s = scale_seq(X_test)

Y_train_s = scaler.transform(Y_train.reshape(-1, 1)).reshape(Y_train.shape).astype(np.float32)
Y_val_s   = scaler.transform(Y_val.reshape(-1, 1)).reshape(Y_val.shape).astype(np.float32)
Y_test_s  = scaler.transform(Y_test.reshape(-1, 1)).reshape(Y_test.shape).astype(np.float32)


Split samples: 247434 79433 41119


In [10]:
class RRSeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_ds = RRSeqDataset(X_train_s, Y_train_s)
val_ds   = RRSeqDataset(X_val_s, Y_val_s)
test_ds  = RRSeqDataset(X_test_s, Y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads, dropout):
        super().__init__()
        self.out_dim = out_dim
        self.heads = heads
        self.W = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, out_dim))
        self.a_dst = nn.Parameter(torch.empty(heads, out_dim))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)
        self.leaky = nn.LeakyReLU(0.2)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):
        # x: (B,N,in_dim)  adj:(N,N)
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.heads, self.out_dim)

        s = (h * self.a_src.view(1,1,self.heads,self.out_dim)).sum(-1)  # (B,N,heads)
        d = (h * self.a_dst.view(1,1,self.heads,self.out_dim)).sum(-1)

        e = self.leaky(s.permute(0,2,1).unsqueeze(-1) + d.permute(0,2,1).unsqueeze(-2))  # (B,heads,N,N)

        mask = (adj == 0).unsqueeze(0).unsqueeze(0)
        e = e.masked_fill(mask, float("-inf"))

        alpha = torch.softmax(e, dim=-1)
        alpha = self.drop(alpha)

        h_head = h.permute(0,2,1,3)   # (B,heads,N,out_dim)
        out = torch.matmul(alpha, h_head)  # (B,heads,N,out_dim)
        out = out.permute(0,2,1,3).contiguous().view(B, N, self.heads*self.out_dim)
        return out

class GATRegressor(nn.Module):
    def __init__(self, N, H, hidden=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(2, hidden)  # RR + dRR
        self.gats = nn.ModuleList()
        for _ in range(layers):
            self.gats.append(GraphAttentionLayer(hidden, hidden//heads, heads, dropout))
        self.act = nn.ELU()
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, H))

        # temporal adjacency
        adj = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            adj[i, i] = 1
            if i-1 >= 0: adj[i, i-1] = 1
            if i+1 < N: adj[i, i+1] = 1
        self.register_buffer("adj", torch.tensor(adj, dtype=torch.float32))

    def forward(self, x):
        rr = x
        drr = torch.zeros_like(rr)
        drr[:, 1:] = rr[:, 1:] - rr[:, :-1]
        feats = torch.stack([rr, drr], dim=-1)  # (B,N,2)

        h = self.in_proj(feats)
        for layer in self.gats:
            h = self.act(layer(h, self.adj))
        h = h.mean(dim=1)
        h = self.drop(h)
        return self.head(h)


In [11]:
class TransformerRegressor(nn.Module):
    def __init__(self, N, H, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.zeros(N, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, H))

    def forward(self, x):
        x = x.unsqueeze(-1)  # (B,N,1)
        z = self.in_proj(x) + self.pos.unsqueeze(0)
        z = self.encoder(z)
        z = z.mean(dim=1)
        z = self.drop(z)
        return self.head(z)


In [14]:
# --- GAT DEFINITIONS (run this cell before training) ---
import torch
import torch.nn as nn
import numpy as np

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads, dropout):
        super().__init__()
        self.out_dim = out_dim
        self.heads = heads
        self.W = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, out_dim))
        self.a_dst = nn.Parameter(torch.empty(heads, out_dim))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)
        self.leaky = nn.LeakyReLU(0.2)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):
        # x: (B,N,in_dim)  adj:(N,N)
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.heads, self.out_dim)

        s = (h * self.a_src.view(1,1,self.heads,self.out_dim)).sum(-1)  # (B,N,heads)
        d = (h * self.a_dst.view(1,1,self.heads,self.out_dim)).sum(-1)

        e = self.leaky(
            s.permute(0,2,1).unsqueeze(-1) + d.permute(0,2,1).unsqueeze(-2)
        )  # (B,heads,N,N)

        mask = (adj == 0).unsqueeze(0).unsqueeze(0)
        e = e.masked_fill(mask, float("-inf"))

        alpha = torch.softmax(e, dim=-1)
        alpha = self.drop(alpha)

        h_head = h.permute(0,2,1,3)   # (B,heads,N,out_dim)
        out = torch.matmul(alpha, h_head)  # (B,heads,N,out_dim)
        out = out.permute(0,2,1,3).contiguous().view(B, N, self.heads*self.out_dim)
        return out

class GATRegressor(nn.Module):
    def __init__(self, N, H, hidden=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        assert hidden % heads == 0, "hidden must be divisible by heads"
        self.in_proj = nn.Linear(2, hidden)  # RR + dRR
        self.gats = nn.ModuleList()
        for _ in range(layers):
            self.gats.append(GraphAttentionLayer(hidden, hidden//heads, heads, dropout))
        self.act = nn.ELU()
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, H))

        # temporal adjacency
        adj = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            adj[i, i] = 1
            if i-1 >= 0: adj[i, i-1] = 1
            if i+1 < N: adj[i, i+1] = 1
        self.register_buffer("adj", torch.tensor(adj, dtype=torch.float32))

    def forward(self, x):
        # x: (B,N)
        rr = x
        drr = torch.zeros_like(rr)
        drr[:, 1:] = rr[:, 1:] - rr[:, :-1]
        feats = torch.stack([rr, drr], dim=-1)  # (B,N,2)

        h = self.in_proj(feats)
        for layer in self.gats:
            h = self.act(layer(h, self.adj))
        h = h.mean(dim=1)  # pool
        h = self.drop(h)
        return self.head(h)

print("✅ GATRegressor is now defined.")


✅ GATRegressor is now defined.


In [15]:
def train_model(model, train_loader, val_loader, epochs=15):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    best = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                pred = model(xb).cpu().numpy()
                ps.append(pred)
                ys.append(yb.numpy())
        y_true = np.concatenate(ys, 0)
        y_pred = np.concatenate(ps, 0)
        val_rmse = rmse(y_true.reshape(-1), y_pred.reshape(-1))

        if val_rmse < best:
            best = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {ep:02d} | train_loss={np.mean(losses):.6f} | val_RMSE_all={val_rmse:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best

def predict_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, 0)


In [16]:
# GAT
gat = GATRegressor(N_WINDOW, H_HORIZON, hidden=64, heads=4, layers=2, dropout=0.1)
gat, val_gat_rmse_all = train_model(gat, train_loader, val_loader, epochs=EPOCHS)
pred_gat_test = predict_model(gat, test_loader)
print("GAT test:", horizon_metrics(Y_test_s, pred_gat_test))

# Transformer
tr = TransformerRegressor(N_WINDOW, H_HORIZON, d_model=64, nhead=4, layers=2, dropout=0.1)
tr, val_tr_rmse_all = train_model(tr, train_loader, val_loader, epochs=EPOCHS)
pred_tr_test = predict_model(tr, test_loader)
print("TR test:", horizon_metrics(Y_test_s, pred_tr_test))


Epoch 01 | train_loss=0.392128 | val_RMSE_all=0.734145
Epoch 02 | train_loss=0.363134 | val_RMSE_all=0.735147
Epoch 03 | train_loss=0.353752 | val_RMSE_all=0.734425
Epoch 04 | train_loss=0.342340 | val_RMSE_all=0.734796
Epoch 05 | train_loss=0.331935 | val_RMSE_all=0.724959
Epoch 06 | train_loss=0.325077 | val_RMSE_all=0.727118
Epoch 07 | train_loss=0.320342 | val_RMSE_all=0.725928
Epoch 08 | train_loss=0.316904 | val_RMSE_all=0.727879
Epoch 09 | train_loss=0.314500 | val_RMSE_all=0.723316
Epoch 10 | train_loss=0.311812 | val_RMSE_all=0.723290
Epoch 11 | train_loss=0.309715 | val_RMSE_all=0.729965
Epoch 12 | train_loss=0.308009 | val_RMSE_all=0.727149
Epoch 13 | train_loss=0.306918 | val_RMSE_all=0.719438
Epoch 14 | train_loss=0.305831 | val_RMSE_all=0.729943
Epoch 15 | train_loss=0.304715 | val_RMSE_all=0.730810
GAT test: {'rmse_k': [0.5186352729797363, 0.5550382137298584, 0.5659667253494263, 0.5738012194633484, 0.5852411985397339], 'mae_k': [0.28702497482299805, 0.3173208236694336, 0

In [18]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

def features_from_window(x):
    diffs = np.diff(x)
    rmssd = np.sqrt(np.mean(diffs**2)) if diffs.size else 0.0
    pnn50 = np.mean(np.abs(diffs) > 0.05) if diffs.size else 0.0
    t = np.arange(len(x), dtype=np.float32)
    slope = np.polyfit(t, x, 1)[0] if len(x) >= 2 else 0.0
    return np.array([
        x[-1],
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        x[-1] - x[-2] if len(x) >= 2 else 0.0,
        rmssd,
        pnn50,
        slope
    ], dtype=np.float32)

def make_xgb_matrix(X_seq):
    return np.stack([features_from_window(x) for x in X_seq], axis=0).astype(np.float32)

X_train_xgb = make_xgb_matrix(X_train_s)
X_val_xgb   = make_xgb_matrix(X_val_s)
X_test_xgb  = make_xgb_matrix(X_test_s)

base_xgb = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

xgb_mo = MultiOutputRegressor(base_xgb)
xgb_mo.fit(X_train_xgb, Y_train_s)

pred_xgb_val  = xgb_mo.predict(X_val_xgb).astype(np.float32)
pred_xgb_test = xgb_mo.predict(X_test_xgb).astype(np.float32)

val_xgb_rmse_all = rmse(Y_val_s.reshape(-1), pred_xgb_val.reshape(-1))
print("✅ XGBoost done.")
print("XGB val RMSE_all:", val_xgb_rmse_all)
print("XGB test:", horizon_metrics(Y_test_s, pred_xgb_test))


✅ XGBoost done.
XGB val RMSE_all: 0.6922186613082886
XGB test: {'rmse_k': [0.4430811405181885, 0.5427095890045166, 0.5564691424369812, 0.5739395618438721, 0.5907993912696838], 'mae_k': [0.1861279159784317, 0.2756211459636688, 0.308185338973999, 0.32746028900146484, 0.3421650528907776], 'rmse_all': 0.5438681840896606, 'mae_all': 0.28791195154190063}


In [19]:
def median_ensemble(p1, p2, p3):
    return np.median(np.stack([p1,p2,p3], axis=0), axis=0).astype(np.float32)

def weighted_ensemble(preds, rmses):
    inv = np.array([1.0/max(r,1e-12) for r in rmses], dtype=np.float64)
    w = inv / inv.sum()
    out = np.zeros_like(preds[0], dtype=np.float64)
    for wi, pi in zip(w, preds):
        out += wi * pi
    return out.astype(np.float32), w.astype(np.float32)

# Median ensemble
pred_med_test = median_ensemble(pred_xgb_test, pred_gat_test, pred_tr_test)
print("MED test:", horizon_metrics(Y_test_s, pred_med_test))

# Weighted average ensemble (weights from validation RMSE_all)
pred_wavg_test, w = weighted_ensemble(
    [pred_xgb_test, pred_gat_test, pred_tr_test],
    [val_xgb_rmse_all, val_gat_rmse_all, val_tr_rmse_all]
)
print("WAVG weights (XGB, GAT, TR):", w)
print("WAVG test:", horizon_metrics(Y_test_s, pred_wavg_test))


MED test: {'rmse_k': [0.4483884572982788, 0.5349660515785217, 0.5532463788986206, 0.5656976699829102, 0.5813788771629333], 'mae_k': [0.2015765756368637, 0.28103208541870117, 0.31063729524612427, 0.3252047002315521, 0.3372739553451538], 'rmse_all': 0.5387648344039917, 'mae_all': 0.2911449074745178}
WAVG weights (XGB, GAT, TR): [0.3354834  0.32279077 0.34172586]
WAVG test: {'rmse_k': [0.44962161779403687, 0.5336944460868835, 0.5501383543014526, 0.5625864267349243, 0.5778858065605164], 'mae_k': [0.2102561742067337, 0.28092309832572937, 0.30888253450393677, 0.3232455849647522, 0.3355567157268524], 'rmse_all': 0.5366743803024292, 'mae_all': 0.2917728126049042}


In [20]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

Y_test_sec = inv_scale(Y_test_s)

xgb_sec = inv_scale(pred_xgb_test)
gat_sec = inv_scale(pred_gat_test)
tr_sec  = inv_scale(pred_tr_test)
med_sec = inv_scale(pred_med_test)
wavg_sec = inv_scale(pred_wavg_test)

print("XGB (sec):", horizon_metrics(Y_test_sec, xgb_sec))
print("GAT (sec):", horizon_metrics(Y_test_sec, gat_sec))
print("TR  (sec):", horizon_metrics(Y_test_sec, tr_sec))
print("MED (sec):", horizon_metrics(Y_test_sec, med_sec))
print("WAV (sec):", horizon_metrics(Y_test_sec, wavg_sec))


XGB (sec): {'rmse_k': [0.059972893446683884, 0.07345801591873169, 0.07532043755054474, 0.07768513262271881, 0.07996717840433121], 'mae_k': [0.025193193927407265, 0.03730647638440132, 0.04171418026089668, 0.04432312771677971, 0.04631347954273224], 'rmse_all': 0.07361484318971634, 'mae_all': 0.038970090448856354}
GAT (sec): {'rmse_k': [0.07019946724176407, 0.0751267522573471, 0.07660597562789917, 0.07766640931367874, 0.07921484857797623], 'mae_k': [0.03885003551840782, 0.04295070469379425, 0.04551422595977783, 0.046712711453437805, 0.04794800281524658], 'rmse_all': 0.07582550495862961, 'mae_all': 0.04439513757824898}
TR  (sec): {'rmse_k': [0.06239815428853035, 0.07351970672607422, 0.07648584246635437, 0.07792642712593079, 0.07997408509254456], 'mae_k': [0.0284045971930027, 0.03933111950755119, 0.043299295008182526, 0.04508035629987717, 0.04660649970173836], 'rmse_all': 0.07431977242231369, 'mae_all': 0.04054437577724457}
MED (sec): {'rmse_k': [0.060691263526678085, 0.07240989804267883, 0

In [21]:
def median_ensemble(p1, p2, p3):
    return np.median(np.stack([p1,p2,p3], axis=0), axis=0).astype(np.float32)

def weighted_ensemble(preds, rmses):
    inv = np.array([1.0/max(r,1e-12) for r in rmses], dtype=np.float64)
    w = inv / inv.sum()
    out = np.zeros_like(preds[0], dtype=np.float64)
    for wi, pi in zip(w, preds):
        out += wi * pi
    return out.astype(np.float32), w.astype(np.float32)

pred_med_test = median_ensemble(pred_xgb_test, pred_gat_test, pred_tr_test)
print("MED test:", horizon_metrics(Y_test_s, pred_med_test))

pred_wavg_test, w = weighted_ensemble(
    [pred_xgb_test, pred_gat_test, pred_tr_test],
    [val_xgb_rmse_all, val_gat_rmse_all, val_tr_rmse_all]
)
print("WAVG weights:", w)
print("WAVG test:", horizon_metrics(Y_test_s, pred_wavg_test))


MED test: {'rmse_k': [0.4483884572982788, 0.5349660515785217, 0.5532463788986206, 0.5656976699829102, 0.5813788771629333], 'mae_k': [0.2015765756368637, 0.28103208541870117, 0.31063729524612427, 0.3252047002315521, 0.3372739553451538], 'rmse_all': 0.5387648344039917, 'mae_all': 0.2911449074745178}
WAVG weights: [0.3354834  0.32279077 0.34172586]
WAVG test: {'rmse_k': [0.44962161779403687, 0.5336944460868835, 0.5501383543014526, 0.5625864267349243, 0.5778858065605164], 'mae_k': [0.2102561742067337, 0.28092309832572937, 0.30888253450393677, 0.3232455849647522, 0.3355567157268524], 'rmse_all': 0.5366743803024292, 'mae_all': 0.2917728126049042}


In [22]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

# Ground truth in seconds
Y_test_sec = inv_scale(Y_test_s)

# Predictions in seconds
xgb_sec  = inv_scale(pred_xgb_test)
gat_sec  = inv_scale(pred_gat_test)
tr_sec   = inv_scale(pred_tr_test)
med_sec  = inv_scale(pred_med_test)
wavg_sec = inv_scale(pred_wavg_test)

print("XGB (seconds):", horizon_metrics(Y_test_sec, xgb_sec))
print("GAT (seconds):", horizon_metrics(Y_test_sec, gat_sec))
print("TR  (seconds):", horizon_metrics(Y_test_sec, tr_sec))
print("MED (seconds):", horizon_metrics(Y_test_sec, med_sec))
print("WAVG (seconds):", horizon_metrics(Y_test_sec, wavg_sec))


XGB (seconds): {'rmse_k': [0.059972893446683884, 0.07345801591873169, 0.07532043755054474, 0.07768513262271881, 0.07996717840433121], 'mae_k': [0.025193193927407265, 0.03730647638440132, 0.04171418026089668, 0.04432312771677971, 0.04631347954273224], 'rmse_all': 0.07361484318971634, 'mae_all': 0.038970090448856354}
GAT (seconds): {'rmse_k': [0.07019946724176407, 0.0751267522573471, 0.07660597562789917, 0.07766640931367874, 0.07921484857797623], 'mae_k': [0.03885003551840782, 0.04295070469379425, 0.04551422595977783, 0.046712711453437805, 0.04794800281524658], 'rmse_all': 0.07582550495862961, 'mae_all': 0.04439513757824898}
TR  (seconds): {'rmse_k': [0.06239815428853035, 0.07351970672607422, 0.07648584246635437, 0.07792642712593079, 0.07997408509254456], 'mae_k': [0.0284045971930027, 0.03933111950755119, 0.043299295008182526, 0.04508035629987717, 0.04660649970173836], 'rmse_all': 0.07431977242231369, 'mae_all': 0.04054437577724457}
MED (seconds): {'rmse_k': [0.060691263526678085, 0.0724

**Persistance Baseline**

In [23]:
# ---------------- Persistence Baseline ----------------
# For each test sample, predict next H RR values as the last RR in the window

# X_test_s shape: (num_samples, N)
last_rr = X_test_s[:, -1]           # (num_samples,)
pred_persist_test = np.repeat(
    last_rr[:, None], H_HORIZON, axis=1
).astype(np.float32)

print("Persistence baseline (scaled):",
      horizon_metrics(Y_test_s, pred_persist_test))

# Convert to seconds
pred_persist_sec = scaler.inverse_transform(
    pred_persist_test.reshape(-1,1)
).reshape(pred_persist_test.shape)

Y_test_sec = scaler.inverse_transform(
    Y_test_s.reshape(-1,1)
).reshape(Y_test_s.shape)

print("Persistence baseline (seconds):",
      horizon_metrics(Y_test_sec, pred_persist_sec))


Persistence baseline (scaled): {'rmse_k': [0.53819739818573, 0.6969146132469177, 0.71468186378479, 0.7097744345664978, 0.7198228240013123], 'mae_k': [0.20556670427322388, 0.3214062452316284, 0.3633652329444885, 0.3803357779979706, 0.39659202098846436], 'rmse_all': 0.679417610168457, 'mae_all': 0.3334532082080841}
Persistence baseline (seconds): {'rmse_k': [0.07284727692604065, 0.0943303108215332, 0.09673518687486649, 0.09607095271348953, 0.09743104130029678], 'mae_k': [0.027824316173791885, 0.04350368306040764, 0.04918300360441208, 0.05148004740476608, 0.05368039384484291], 'rmse_all': 0.09196202456951141, 'mae_all': 0.045134287327528}


**LSTM**

In [24]:
import torch
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=1, horizon=5, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, horizon)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)          # (B, N, 1)
        out, (h, c) = self.lstm(x)   # h: (num_layers, B, hidden_dim)
        h_last = h[-1]               # (B, hidden_dim)
        return self.head(h_last)     # (B, H)


In [25]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, wd=1e-5):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()

    best = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                ps.append(model(xb).cpu().numpy())
                ys.append(yb.numpy())
        y_true = np.concatenate(ys, 0)
        y_pred = np.concatenate(ps, 0)
        val_rmse_all = rmse(y_true.reshape(-1), y_pred.reshape(-1))

        if val_rmse_all < best:
            best = val_rmse_all
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {ep:02d} | train_loss={np.mean(losses):.6f} | val_RMSE_all={val_rmse_all:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best

def predict_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, 0)


In [26]:
from torch.utils.data import Dataset, DataLoader

class RRSeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

train_ds = RRSeqDataset(X_train_s, Y_train_s)
val_ds   = RRSeqDataset(X_val_s, Y_val_s)
test_ds  = RRSeqDataset(X_test_s, Y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [27]:
lstm = LSTMRegressor(input_dim=1, hidden_dim=64, num_layers=1, horizon=H_HORIZON, dropout=0.0)
lstm, val_lstm_rmse_all = train_model(lstm, train_loader, val_loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY)

pred_lstm_test = predict_model(lstm, test_loader)

print("LSTM val RMSE_all (scaled):", val_lstm_rmse_all)
print("LSTM test (scaled):", horizon_metrics(Y_test_s, pred_lstm_test))


Epoch 01 | train_loss=0.296060 | val_RMSE_all=0.695663
Epoch 02 | train_loss=0.252397 | val_RMSE_all=0.678741
Epoch 03 | train_loss=0.242892 | val_RMSE_all=0.679234
Epoch 04 | train_loss=0.237984 | val_RMSE_all=0.680678
Epoch 05 | train_loss=0.233940 | val_RMSE_all=0.679098
Epoch 06 | train_loss=0.230948 | val_RMSE_all=0.677101
Epoch 07 | train_loss=0.227937 | val_RMSE_all=0.678407
Epoch 08 | train_loss=0.224905 | val_RMSE_all=0.680507
Epoch 09 | train_loss=0.221991 | val_RMSE_all=0.681189
Epoch 10 | train_loss=0.219391 | val_RMSE_all=0.686792
Epoch 11 | train_loss=0.217017 | val_RMSE_all=0.691019
Epoch 12 | train_loss=0.214603 | val_RMSE_all=0.687144
Epoch 13 | train_loss=0.212327 | val_RMSE_all=0.696031
Epoch 14 | train_loss=0.210139 | val_RMSE_all=0.703287
Epoch 15 | train_loss=0.208169 | val_RMSE_all=0.697130
LSTM val RMSE_all (scaled): 0.6771007180213928
LSTM test (scaled): {'rmse_k': [0.45841294527053833, 0.5362170934677124, 0.5563740730285645, 0.5693429708480835, 0.5858919620513

In [28]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

Y_test_sec = inv_scale(Y_test_s)
lstm_sec = inv_scale(pred_lstm_test)

print("LSTM test (seconds):", horizon_metrics(Y_test_sec, lstm_sec))


LSTM test (seconds): {'rmse_k': [0.06204811483621597, 0.07257923483848572, 0.07530757039785385, 0.07706295698881149, 0.07930293679237366], 'mae_k': [0.02731219306588173, 0.03817078843712807, 0.04272295907139778, 0.04487698897719383, 0.04683585464954376], 'rmse_all': 0.07350727915763855, 'mae_all': 0.039983753114938736}


**Statistical significance 5 runs mean ± std paired test (p-value)**

In [29]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [30]:
from scipy.stats import ttest_rel

SEEDS = [0, 1, 2, 3, 4]

rmse_lstm_runs = []
rmse_ens_runs = []

for seed in SEEDS:
    print(f"\n===== RUN seed={seed} =====")
    set_seed(seed)

    # ---- LSTM ----
    lstm = LSTMRegressor(
        input_dim=1,
        hidden_dim=64,
        num_layers=1,
        horizon=H_HORIZON
    )
    lstm, _ = train_model(lstm, train_loader, val_loader, epochs=EPOCHS)
    pred_lstm = predict_model(lstm, test_loader)

    lstm_sec = scaler.inverse_transform(
        pred_lstm.reshape(-1,1)
    ).reshape(pred_lstm.shape)

    Y_test_sec = scaler.inverse_transform(
        Y_test_s.reshape(-1,1)
    ).reshape(Y_test_s.shape)

    rmse_lstm = rmse(Y_test_sec.reshape(-1), lstm_sec.reshape(-1))
    rmse_lstm_runs.append(rmse_lstm)

    # ---- ENSEMBLE (use your already-trained components or retrain if needed) ----
    # If retraining everything is too slow, retrain only Transformer+GAT once and reuse XGB

    pred_ens = inv_scale(pred_wavg_test)  # OR recompute per run if retraining
    rmse_ens = rmse(Y_test_sec.reshape(-1), pred_ens.reshape(-1))
    rmse_ens_runs.append(rmse_ens)

    print(f"LSTM RMSE: {rmse_lstm:.5f}, Ensemble RMSE: {rmse_ens:.5f}")



===== RUN seed=0 =====
Epoch 01 | train_loss=0.294353 | val_RMSE_all=0.689431
Epoch 02 | train_loss=0.249556 | val_RMSE_all=0.679408
Epoch 03 | train_loss=0.240200 | val_RMSE_all=0.683069
Epoch 04 | train_loss=0.234619 | val_RMSE_all=0.682058
Epoch 05 | train_loss=0.230876 | val_RMSE_all=0.679614
Epoch 06 | train_loss=0.227297 | val_RMSE_all=0.681543
Epoch 07 | train_loss=0.224252 | val_RMSE_all=0.685986
Epoch 08 | train_loss=0.221083 | val_RMSE_all=0.688356
Epoch 09 | train_loss=0.218757 | val_RMSE_all=0.691339
Epoch 10 | train_loss=0.216131 | val_RMSE_all=0.690909
Epoch 11 | train_loss=0.213894 | val_RMSE_all=0.703765
Epoch 12 | train_loss=0.212208 | val_RMSE_all=0.700303
Epoch 13 | train_loss=0.210171 | val_RMSE_all=0.703478
Epoch 14 | train_loss=0.208306 | val_RMSE_all=0.711087
Epoch 15 | train_loss=0.206375 | val_RMSE_all=0.715178
LSTM RMSE: 0.07350, Ensemble RMSE: 0.07264

===== RUN seed=1 =====
Epoch 01 | train_loss=0.293137 | val_RMSE_all=0.690447
Epoch 02 | train_loss=0.25302

In [31]:
import numpy as np

lstm_mean = np.mean(rmse_lstm_runs)
lstm_std  = np.std(rmse_lstm_runs)

ens_mean = np.mean(rmse_ens_runs)
ens_std  = np.std(rmse_ens_runs)

print(f"LSTM:     {lstm_mean:.4f} ± {lstm_std:.4f}")
print(f"Ensemble: {ens_mean:.4f} ± {ens_std:.4f}")


LSTM:     0.0735 ± 0.0005
Ensemble: 0.0726 ± 0.0000


In [32]:
t_stat, p_value = ttest_rel(rmse_lstm_runs, rmse_ens_runs)

print("Paired t-test p-value:", p_value)

Paired t-test p-value: 0.0258403973550514


**code for Abnormal RR detection**

In [33]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

# Residual scores
residual = np.abs(Y_test_sec - wavg_sec)   # (N,H)

score_h1 = residual[:, 0]                  # horizon 1
score_mean = residual.mean(axis=1)         # average error across horizons
score_max = residual.max(axis=1)           # max error across horizons


In [34]:
X_test_sec = inv_scale(X_test_s)  # shape (N, window)
rr_next_true = Y_test_sec[:, 0]   # RR(t+1)


In [35]:
eps = 1e-8
med = np.median(X_test_sec, axis=1)
mad = np.median(np.abs(X_test_sec - med[:, None]), axis=1)
robust_z = np.abs(rr_next_true - med) / (1.4826 * mad + eps)

TAU = 3.5
y_abn = (robust_z > TAU).astype(int)  # 1=abnormal, 0=normal
print("Abnormal rate:", y_abn.mean())


Abnormal rate: 0.044577932342712616


In [37]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

# Validation in seconds
X_val_sec = inv_scale(X_val_s)     # (N_val, 30)
Y_val_sec = inv_scale(Y_val_s)     # (N_val, 5)

# ---- Build WAVG prediction for VAL ----
# First: get model predictions on VAL (scaled)
pred_xgb_val = xgb_mo.predict(make_xgb_matrix(X_val_s)).astype(np.float32)
pred_gat_val = predict_model(gat, val_loader).astype(np.float32)
pred_tr_val  = predict_model(tr,  val_loader).astype(np.float32)

# Compute WAVG weights (from validation RMSE of each model on VAL)
val_xgb_rmse = rmse(Y_val_s.reshape(-1), pred_xgb_val.reshape(-1))
val_gat_rmse = rmse(Y_val_s.reshape(-1), pred_gat_val.reshape(-1))
val_tr_rmse  = rmse(Y_val_s.reshape(-1), pred_tr_val.reshape(-1))

inv = np.array([1.0/max(val_xgb_rmse,1e-12), 1.0/max(val_gat_rmse,1e-12), 1.0/max(val_tr_rmse,1e-12)])
w = inv / inv.sum()
print("VAL weights (XGB, GAT, TR):", w)

pred_wavg_val = (w[0]*pred_xgb_val + w[1]*pred_gat_val + w[2]*pred_tr_val).astype(np.float32)

# Convert WAVG VAL prediction to seconds
wavg_val_sec = inv_scale(pred_wavg_val)

print("Shapes:", X_val_sec.shape, Y_val_sec.shape, wavg_val_sec.shape)


VAL weights (XGB, GAT, TR): [0.33548339 0.32279076 0.34172585]
Shapes: (79433, 30) (79433, 5) (79433, 5)


In [38]:
from sklearn.metrics import f1_score, average_precision_score, precision_score, recall_score

# Residual score on VAL (mean across horizons)
val_res = np.abs(Y_val_sec - wavg_val_sec)
val_score = val_res.mean(axis=1)

# Abnormal labels using robust MAD rule on horizon-1 truth
eps = 1e-8
med = np.median(X_val_sec, axis=1)
mad = np.median(np.abs(X_val_sec - med[:, None]), axis=1)
robust_z_val = np.abs(Y_val_sec[:,0] - med) / (1.4826*mad + eps)

TAU = 3.5
y_val_abn = (robust_z_val > TAU).astype(int)
print("VAL abnormal rate:", y_val_abn.mean())

# Search threshold that maximizes F1 on VAL
ths = np.linspace(np.percentile(val_score, 50), np.percentile(val_score, 99.5), 120)

best_f1, best_th = -1, None
for th in ths:
    pred = (val_score >= th).astype(int)
    f1 = f1_score(y_val_abn, pred, zero_division=0)
    if f1 > best_f1:
        best_f1, best_th = f1, th

print("Best VAL threshold:", best_th)
print("Best VAL F1:", best_f1)
print("VAL AUPRC:", average_precision_score(y_val_abn, val_score))


VAL abnormal rate: 0.04542192791409112
Best VAL threshold: 0.08224423664958526
Best VAL F1: 0.15149132463175552
VAL AUPRC: 0.09596024215637741


In [39]:
# TEST in seconds (if not already created)
X_test_sec = inv_scale(X_test_s)
Y_test_sec = inv_scale(Y_test_s)
wavg_sec   = inv_scale(pred_wavg_test)

# Residual score on TEST
test_res = np.abs(Y_test_sec - wavg_sec)
test_score = test_res.mean(axis=1)

# TEST abnormal labels using same MAD rule
eps = 1e-8
med_t = np.median(X_test_sec, axis=1)
mad_t = np.median(np.abs(X_test_sec - med_t[:, None]), axis=1)
robust_z_test = np.abs(Y_test_sec[:,0] - med_t) / (1.4826*mad_t + eps)

y_test_abn = (robust_z_test > TAU).astype(int)
print("TEST abnormal rate:", y_test_abn.mean())

# Predict abnormal using threshold from VAL
y_pred = (test_score >= best_th).astype(int)

F1 = f1_score(y_test_abn, y_pred, zero_division=0)
P  = precision_score(y_test_abn, y_pred, zero_division=0)
R  = recall_score(y_test_abn, y_pred, zero_division=0)
AUPRC = average_precision_score(y_test_abn, test_score)

print("Ensemble residual abnormal RR detection (TEST)")
print("F1:", F1, "Precision:", P, "Recall:", R, "AUPRC:", AUPRC)


TEST abnormal rate: 0.044577932342712616
Ensemble residual abnormal RR detection (TEST)
F1: 0.3064159292035398 Precision: 0.23141186299081035 Recall: 0.4533551554828151 AUPRC: 0.2325625032801071


In [40]:
raw_score = robust_z_test
print("RAW outlier AUPRC:", average_precision_score(y_test_abn, raw_score))


RAW outlier AUPRC: 1.0


In [41]:
last_rr = X_test_sec[:, -1]
persist_pred = np.repeat(last_rr[:,None], H_HORIZON, axis=1).astype(np.float32)

persist_score = np.abs(Y_test_sec - persist_pred).mean(axis=1)
print("Persistence residual AUPRC:", average_precision_score(y_test_abn, persist_score))


Persistence residual AUPRC: 0.23359515076994397


**Cross dataset**

In [42]:
# ============================================================
# FULL KAGGLE NOTEBOOK CODE (TRAIN ON MIT-Poly, TEST ON PTB-XL)
# Models: Persistence, LSTM, XGBoost, GAT, Transformer
# Ensembles: Median + Weighted Average
# Cross-dataset: Train on MIT only, evaluate zero-shot on PTB-XL
# ============================================================

import os, random, glob
import numpy as np
from tqdm import tqdm

import wfdb
import neurokit2 as nk
from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---------------- CONFIG ----------------
SEED = 42
N_WINDOW = 30
H_HORIZON = 5
RR_MIN, RR_MAX = 0.25, 2.50

# ECG preprocessing
BP_LOW, BP_HIGH = 0.5, 40.0
BP_ORDER = 3

# split (record-wise)
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# training
BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-5

# Paths (EDIT THESE IF NEEDED)
MIT_DIR = "/kaggle/input/final-mith-dataset/mit-bih-polysomnographic-database-1.0.0"
PTBXL_DIR = "/kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# Optional speed limit for PTB-XL (set None for full)
PTBXL_MAX_RECORDS = None  # e.g., 50 for quick test

# ---------------- SEED ----------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ---------------- METRICS ----------------
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean(np.abs(y_true - y_pred)))

def horizon_metrics(Y_true, Y_pred):
    Y_true = np.asarray(Y_true)
    Y_pred = np.asarray(Y_pred)
    rmse_k = [rmse(Y_true[:,k], Y_pred[:,k]) for k in range(Y_true.shape[1])]
    mae_k  = [mae (Y_true[:,k], Y_pred[:,k]) for k in range(Y_true.shape[1])]
    return {
        "rmse_k": rmse_k,
        "mae_k": mae_k,
        "rmse_all": rmse(Y_true.reshape(-1), Y_pred.reshape(-1)),
        "mae_all":  mae (Y_true.reshape(-1), Y_pred.reshape(-1)),
    }

def report_metrics_seconds(name, Y_sec, pred_sec):
    m = horizon_metrics(Y_sec, pred_sec)
    print(f"\n=== {name} ===")
    print("RMSE_all (s):", m["rmse_all"], " MAE_all (s):", m["mae_all"])
    print("RMSE_k (s):", m["rmse_k"])
    print("MAE_k  (s):", m["mae_k"])
    print("RMSE_all (ms):", m["rmse_all"]*1000.0, " MAE_all (ms):", m["mae_all"]*1000.0)
    return m

# ---------------- SIGNAL UTILS ----------------
def bandpass_filter(x, fs, low, high, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, x).astype(np.float32)

def build_windows(rr, N, H):
    rr = np.asarray(rr, dtype=np.float32)
    L = len(rr)
    if L < N + H:
        return np.empty((0,N), np.float32), np.empty((0,H), np.float32)
    X = []
    Y = []
    for i in range(N, L - H):
        X.append(rr[i-N:i])
        Y.append(rr[i:i+H])
    return np.stack(X).astype(np.float32), np.stack(Y).astype(np.float32)

def pick_ecg_channel(sig_names):
    for i, name in enumerate(sig_names):
        if "ecg" in name.lower():
            return i
    # fallback: sometimes lead names like "II"
    for i, name in enumerate(sig_names):
        if name.strip().upper() in ["II", "I", "V1", "V2", "V3", "V4", "V5", "V6"]:
            return i
    return 0

# ---------------- RECORD LISTING ----------------
def read_records_list(data_dir):
    with open(os.path.join(data_dir, "RECORDS"), "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def list_wfdb_records_anywhere(root_dir):
    records_path = os.path.join(root_dir, "RECORDS")
    if os.path.exists(records_path):
        with open(records_path, "r", encoding="utf-8") as f:
            recs = [line.strip() for line in f if line.strip()]
        return [os.path.join(root_dir, r) for r in recs]
    hea_files = glob.glob(os.path.join(root_dir, "**", "*.hea"), recursive=True)
    bases = sorted(list(set([hf[:-4] for hf in hea_files])))
    return bases

# ---------------- RR EXTRACTION ----------------
def extract_rr_from_record_base(record_base):
    # record_base is full base path without extension
    try:
        rec = wfdb.rdrecord(record_base)
        fs = float(rec.fs)
        sig = rec.p_signal
        if sig is None:
            return None

        ch_idx = pick_ecg_channel(rec.sig_name)
        ecg = sig[:, ch_idx].astype(np.float32)

        ecg_f = bandpass_filter(ecg, fs, BP_LOW, BP_HIGH, BP_ORDER)

        _, info = nk.ecg_peaks(ecg_f, sampling_rate=fs)
        rpeaks = info.get("ECG_R_P_Peaks", None)  # typo-safe
        if rpeaks is None:
            rpeaks = info.get("ECG_R_Peaks", None)
        if rpeaks is None or len(rpeaks) < (N_WINDOW + H_HORIZON + 10):
            return None

        rpeaks = np.asarray(rpeaks, dtype=np.int64)
        rr = (np.diff(rpeaks) / fs).astype(np.float32)
        rr = rr[(rr >= RR_MIN) & (rr <= RR_MAX)]
        if rr.size < (N_WINDOW + H_HORIZON + 10):
            return None
        return rr
    except Exception:
        return None

def build_rr_windows_from_record_bases(record_bases, max_records=None):
    X_all, Y_all, rec_ids = [], [], []
    used = 0
    for base in tqdm(record_bases):
        if (max_records is not None) and (used >= max_records):
            break
        rr = extract_rr_from_record_base(base)
        if rr is None:
            continue
        Xr, Yr = build_windows(rr, N_WINDOW, H_HORIZON)
        if Xr.size == 0:
            continue
        X_all.append(Xr); Y_all.append(Yr)
        rec_ids.extend([os.path.basename(base)] * Xr.shape[0])
        used += 1
    if len(X_all) == 0:
        return None, None, None
    return np.concatenate(X_all,0), np.concatenate(Y_all,0), np.array(rec_ids)

# ---------------- DATASET / LOADER ----------------
class RRSeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

# ---------------- MODELS ----------------
class TransformerRegressor(nn.Module):
    def __init__(self, N, H, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.zeros(N, d_model))
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, H))

    def forward(self, x):
        x = x.unsqueeze(-1)  # (B,N,1)
        z = self.in_proj(x) + self.pos.unsqueeze(0)
        z = self.encoder(z)
        z = z.mean(dim=1)
        z = self.drop(z)
        return self.head(z)

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads, dropout):
        super().__init__()
        self.out_dim = out_dim
        self.heads = heads
        self.W = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, out_dim))
        self.a_dst = nn.Parameter(torch.empty(heads, out_dim))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)
        self.leaky = nn.LeakyReLU(0.2)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, adj):
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.heads, self.out_dim)

        s = (h * self.a_src.view(1,1,self.heads,self.out_dim)).sum(-1)
        d = (h * self.a_dst.view(1,1,self.heads,self.out_dim)).sum(-1)

        e = self.leaky(
            s.permute(0,2,1).unsqueeze(-1) + d.permute(0,2,1).unsqueeze(-2)
        )  # (B,heads,N,N)

        mask = (adj == 0).unsqueeze(0).unsqueeze(0)
        e = e.masked_fill(mask, float("-inf"))

        alpha = torch.softmax(e, dim=-1)
        alpha = self.drop(alpha)

        h_head = h.permute(0,2,1,3)
        out = torch.matmul(alpha, h_head)
        out = out.permute(0,2,1,3).contiguous().view(B, N, self.heads*self.out_dim)
        return out

class GATRegressor(nn.Module):
    def __init__(self, N, H, hidden=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        assert hidden % heads == 0
        self.in_proj = nn.Linear(2, hidden)  # RR + dRR
        self.gats = nn.ModuleList([
            GraphAttentionLayer(hidden, hidden//heads, heads, dropout) for _ in range(layers)
        ])
        self.act = nn.ELU()
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, H))

        adj = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            adj[i,i] = 1
            if i-1 >= 0: adj[i,i-1] = 1
            if i+1 < N: adj[i,i+1] = 1
        self.register_buffer("adj", torch.tensor(adj, dtype=torch.float32))

    def forward(self, x):
        rr = x
        drr = torch.zeros_like(rr)
        drr[:,1:] = rr[:,1:] - rr[:,:-1]
        feats = torch.stack([rr, drr], dim=-1)  # (B,N,2)

        h = self.in_proj(feats)
        for layer in self.gats:
            h = self.act(layer(h, self.adj))
        h = h.mean(dim=1)
        h = self.drop(h)
        return self.head(h)

class LSTMRegressor(nn.Module):
    def __init__(self, horizon=5, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=0.0)
        self.head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, horizon))

    def forward(self, x):
        x = x.unsqueeze(-1)  # (B,N,1)
        out, (h, c) = self.lstm(x)
        h_last = h[-1]
        return self.head(h_last)

# ---------------- TRAIN / PREDICT ----------------
def train_model(model, train_loader, val_loader, epochs=15):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                ps.append(model(xb).cpu().numpy())
                ys.append(yb.numpy())
        y_true = np.concatenate(ys,0)
        y_pred = np.concatenate(ps,0)
        val_rmse_all = rmse(y_true.reshape(-1), y_pred.reshape(-1))

        if val_rmse_all < best:
            best = val_rmse_all
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

        print(f"Epoch {ep:02d} | train_loss={np.mean(losses):.6f} | val_RMSE_all={val_rmse_all:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best

def predict_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds,0).astype(np.float32)

# ---------------- XGBOOST FEATURES ----------------
def features_from_window(x):
    diffs = np.diff(x)
    rmssd = np.sqrt(np.mean(diffs**2)) if diffs.size else 0.0
    pnn50 = np.mean(np.abs(diffs) > 0.05) if diffs.size else 0.0
    t = np.arange(len(x), dtype=np.float32)
    slope = np.polyfit(t, x, 1)[0] if len(x) >= 2 else 0.0
    return np.array([
        x[-1],
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        x[-1] - x[-2] if len(x) >= 2 else 0.0,
        rmssd,
        pnn50,
        slope
    ], dtype=np.float32)

def make_xgb_matrix(X_seq):
    return np.stack([features_from_window(x) for x in X_seq], axis=0).astype(np.float32)

# ---------------- ENSEMBLES ----------------
def median_ensemble(p1, p2, p3):
    return np.median(np.stack([p1,p2,p3], axis=0), axis=0).astype(np.float32)

def weighted_ensemble(preds, rmses):
    inv = np.array([1.0/max(r,1e-12) for r in rmses], dtype=np.float64)
    w = inv / inv.sum()
    out = np.zeros_like(preds[0], dtype=np.float64)
    for wi, pi in zip(w, preds):
        out += wi * pi
    return out.astype(np.float32), w.astype(np.float32)

# ---------------- SCALE UTILS ----------------
def scale_with_train_scaler(scaler, X):
    return scaler.transform(X.reshape(-1,1)).reshape(X.shape).astype(np.float32)

def inv_scale_with_train_scaler(scaler, Xs):
    return scaler.inverse_transform(Xs.reshape(-1,1)).reshape(Xs.shape).astype(np.float32)

def persistence_pred_sec(X_sec):
    last_rr = X_sec[:, -1]
    return np.repeat(last_rr[:,None], H_HORIZON, axis=1).astype(np.float32)

# ============================================================
# 1) LOAD + WINDOW MIT (TRAIN DOMAIN)
# ============================================================
print("MIT exists:", os.path.exists(MIT_DIR))
print("Has RECORDS:", os.path.exists(os.path.join(MIT_DIR, "RECORDS")))

mit_records_rel = read_records_list(MIT_DIR)
mit_record_bases = [os.path.join(MIT_DIR, r) for r in mit_records_rel]
print("MIT total records:", len(mit_record_bases), "example:", mit_record_bases[:3])

X_all, Y_all, rec_ids = build_rr_windows_from_record_bases(mit_record_bases)
print("MIT samples:", X_all.shape, "targets:", Y_all.shape)
unique_recs = np.unique(rec_ids)
print("MIT records used:", len(unique_recs))

# Record-wise split
train_recs, test_recs = train_test_split(unique_recs, test_size=TEST_SIZE, random_state=SEED)
val_frac_of_train = VAL_SIZE / (1.0 - TEST_SIZE)
train_recs, val_recs = train_test_split(train_recs, test_size=val_frac_of_train, random_state=SEED)

m_train = np.isin(rec_ids, train_recs)
m_val   = np.isin(rec_ids, val_recs)
m_test  = np.isin(rec_ids, test_recs)

X_train, Y_train = X_all[m_train], Y_all[m_train]
X_val,   Y_val   = X_all[m_val],   Y_all[m_val]
X_test,  Y_test  = X_all[m_test],  Y_all[m_test]
print("MIT split samples:", X_train.shape, X_val.shape, X_test.shape)

# Fit scaler ONLY on MIT train RR values
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1,1))

X_train_s = scale_with_train_scaler(scaler, X_train)
X_val_s   = scale_with_train_scaler(scaler, X_val)
X_test_s  = scale_with_train_scaler(scaler, X_test)

Y_train_s = scale_with_train_scaler(scaler, Y_train)
Y_val_s   = scale_with_train_scaler(scaler, Y_val)
Y_test_s  = scale_with_train_scaler(scaler, Y_test)

train_loader = DataLoader(RRSeqDataset(X_train_s, Y_train_s), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RRSeqDataset(X_val_s,   Y_val_s),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(RRSeqDataset(X_test_s,  Y_test_s),  batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 2) TRAIN MODELS ON MIT ONLY
# ============================================================

# ---- Persistence baseline (MIT TEST) ----
X_test_sec = inv_scale_with_train_scaler(scaler, X_test_s)
Y_test_sec = inv_scale_with_train_scaler(scaler, Y_test_s)
pred_persist_sec = persistence_pred_sec(X_test_sec)
report_metrics_seconds("MIT TEST - Persistence", Y_test_sec, pred_persist_sec)

# ---- LSTM baseline ----
lstm = LSTMRegressor(horizon=H_HORIZON, hidden_dim=64, num_layers=1)
lstm, val_lstm_rmse = train_model(lstm, train_loader, val_loader, epochs=EPOCHS)
pred_lstm_test_s = predict_model(lstm, test_loader)
pred_lstm_test_sec = inv_scale_with_train_scaler(scaler, pred_lstm_test_s)
report_metrics_seconds("MIT TEST - LSTM", Y_test_sec, pred_lstm_test_sec)

# ---- GAT ----
gat = GATRegressor(N_WINDOW, H_HORIZON, hidden=64, heads=4, layers=2, dropout=0.1)
gat, val_gat_rmse = train_model(gat, train_loader, val_loader, epochs=EPOCHS)
pred_gat_test_s = predict_model(gat, test_loader)

# ---- Transformer ----
tr = TransformerRegressor(N_WINDOW, H_HORIZON, d_model=64, nhead=4, layers=2, dropout=0.1)
tr, val_tr_rmse = train_model(tr, train_loader, val_loader, epochs=EPOCHS)
pred_tr_test_s = predict_model(tr, test_loader)

# ---- XGBoost ----
X_train_xgb = make_xgb_matrix(X_train_s)
X_val_xgb   = make_xgb_matrix(X_val_s)
X_test_xgb  = make_xgb_matrix(X_test_s)

base_xgb = xgb.XGBRegressor(
    n_estimators=800, max_depth=6, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective="reg:squarederror", random_state=SEED, n_jobs=-1
)
xgb_mo = MultiOutputRegressor(base_xgb)
xgb_mo.fit(X_train_xgb, Y_train_s)

pred_xgb_val_s  = xgb_mo.predict(X_val_xgb).astype(np.float32)
pred_xgb_test_s = xgb_mo.predict(X_test_xgb).astype(np.float32)
val_xgb_rmse = rmse(Y_val_s.reshape(-1), pred_xgb_val_s.reshape(-1))

# ---- Ensembles (weights from MIT validation RMSE) ----
pred_med_test_s = median_ensemble(pred_xgb_test_s, pred_gat_test_s, pred_tr_test_s)
pred_wavg_test_s, w = weighted_ensemble(
    [pred_xgb_test_s, pred_gat_test_s, pred_tr_test_s],
    [val_xgb_rmse, val_gat_rmse, val_tr_rmse]
)
print("\nMIT validation RMSE_all:", {"xgb": val_xgb_rmse, "gat": val_gat_rmse, "tr": val_tr_rmse})
print("MIT ensemble weights (xgb,gat,tr):", w)

# Convert MIT test preds to seconds + report
pred_xgb_test_sec  = inv_scale_with_train_scaler(scaler, pred_xgb_test_s)
pred_gat_test_sec  = inv_scale_with_train_scaler(scaler, pred_gat_test_s)
pred_tr_test_sec   = inv_scale_with_train_scaler(scaler, pred_tr_test_s)
pred_med_test_sec  = inv_scale_with_train_scaler(scaler, pred_med_test_s)
pred_wavg_test_sec = inv_scale_with_train_scaler(scaler, pred_wavg_test_s)

report_metrics_seconds("MIT TEST - XGBoost", Y_test_sec, pred_xgb_test_sec)
report_metrics_seconds("MIT TEST - GAT", Y_test_sec, pred_gat_test_sec)
report_metrics_seconds("MIT TEST - Transformer", Y_test_sec, pred_tr_test_sec)
report_metrics_seconds("MIT TEST - Median Ensemble", Y_test_sec, pred_med_test_sec)
report_metrics_seconds("MIT TEST - Weighted Ensemble", Y_test_sec, pred_wavg_test_sec)

# ============================================================
# 3) CROSS-DATASET EVALUATION: PTB-XL (ZERO-SHOT)
# ============================================================
print("\nPTB-XL exists:", os.path.exists(PTBXL_DIR))
print("Listing /kaggle/input:")
!ls /kaggle/input

ptb_record_bases = list_wfdb_records_anywhere(PTBXL_DIR)
print("PTB record bases found:", len(ptb_record_bases), "example:", ptb_record_bases[:3])

X_ptb, Y_ptb, ptb_rec_ids = build_rr_windows_from_record_bases(ptb_record_bases, max_records=PTBXL_MAX_RECORDS)

if X_ptb is None:
    raise RuntimeError("No PTB windows created. Check PTBXL_DIR or ECG channel selection or R-peak extraction.")

print("PTB windows:", X_ptb.shape, "targets:", Y_ptb.shape)
print("PTB records used:", len(np.unique(ptb_rec_ids)))

# Scale PTB using MIT scaler (DO NOT FIT NEW SCALER)
X_ptb_s = scale_with_train_scaler(scaler, X_ptb)
Y_ptb_s = scale_with_train_scaler(scaler, Y_ptb)

ptb_loader = DataLoader(RRSeqDataset(X_ptb_s, Y_ptb_s), batch_size=BATCH_SIZE, shuffle=False)

# Predict PTB (scaled) using MIT-trained models
pred_ptb_gat_s = predict_model(gat, ptb_loader)
pred_ptb_tr_s  = predict_model(tr,  ptb_loader)

X_ptb_xgb = make_xgb_matrix(X_ptb_s)
pred_ptb_xgb_s = xgb_mo.predict(X_ptb_xgb).astype(np.float32)

# LSTM cross-dataset (optional but recommended)
pred_ptb_lstm_s = predict_model(lstm, ptb_loader)

# Ensembles with fixed MIT weights
pred_ptb_wavg_s = (w[0]*pred_ptb_xgb_s + w[1]*pred_ptb_gat_s + w[2]*pred_ptb_tr_s).astype(np.float32)
pred_ptb_med_s  = median_ensemble(pred_ptb_xgb_s, pred_ptb_gat_s, pred_ptb_tr_s)

# Convert PTB to seconds using MIT scaler
X_ptb_sec = inv_scale_with_train_scaler(scaler, X_ptb_s)
Y_ptb_sec = inv_scale_with_train_scaler(scaler, Y_ptb_s)

pred_ptb_persist_sec = persistence_pred_sec(X_ptb_sec)
pred_ptb_xgb_sec     = inv_scale_with_train_scaler(scaler, pred_ptb_xgb_s)
pred_ptb_gat_sec     = inv_scale_with_train_scaler(scaler, pred_ptb_gat_s)
pred_ptb_tr_sec      = inv_scale_with_train_scaler(scaler, pred_ptb_tr_s)
pred_ptb_lstm_sec    = inv_scale_with_train_scaler(scaler, pred_ptb_lstm_s)
pred_ptb_med_sec     = inv_scale_with_train_scaler(scaler, pred_ptb_med_s)
pred_ptb_wavg_sec    = inv_scale_with_train_scaler(scaler, pred_ptb_wavg_s)

# Report PTB-XL results
report_metrics_seconds("PTB-XL TEST - Persistence", Y_ptb_sec, pred_ptb_persist_sec)
report_metrics_seconds("PTB-XL TEST - LSTM (MIT-trained)", Y_ptb_sec, pred_ptb_lstm_sec)
report_metrics_seconds("PTB-XL TEST - XGBoost (MIT-trained)", Y_ptb_sec, pred_ptb_xgb_sec)
report_metrics_seconds("PTB-XL TEST - GAT (MIT-trained)", Y_ptb_sec, pred_ptb_gat_sec)
report_metrics_seconds("PTB-XL TEST - Transformer (MIT-trained)", Y_ptb_sec, pred_ptb_tr_sec)
report_metrics_seconds("PTB-XL TEST - Median Ensemble (MIT-trained)", Y_ptb_sec, pred_ptb_med_sec)
report_metrics_seconds("PTB-XL TEST - Weighted Ensemble (MIT-trained)", Y_ptb_sec, pred_ptb_wavg_sec)

print("\nDONE ✅")


torch: 2.6.0+cu124
DEVICE: cuda
MIT exists: True
Has RECORDS: True
MIT total records: 18 example: ['/kaggle/input/final-mith-dataset/mit-bih-polysomnographic-database-1.0.0/slp01a', '/kaggle/input/final-mith-dataset/mit-bih-polysomnographic-database-1.0.0/slp01b', '/kaggle/input/final-mith-dataset/mit-bih-polysomnographic-database-1.0.0/slp02a']


100%|██████████| 18/18 [00:25<00:00,  1.41s/it]


MIT samples: (367968, 30) targets: (367968, 5)
MIT records used: 18
MIT split samples: (247422, 30) (79430, 30) (41116, 30)

=== MIT TEST - Persistence ===
RMSE_all (s): 0.09196484833955765  MAE_all (s): 0.045135676860809326
RMSE_k (s): [0.07284948229789734, 0.09433314204216003, 0.09673760831356049, 0.09607408940792084, 0.09743449091911316]
MAE_k  (s): [0.02782469242811203, 0.04350452125072479, 0.04918377101421356, 0.0514819510281086, 0.053683433681726456]
RMSE_all (ms): 91.96484833955765  MAE_all (ms): 45.135676860809326
Epoch 01 | train_loss=0.294820 | val_RMSE_all=0.685356
Epoch 02 | train_loss=0.250544 | val_RMSE_all=0.682455
Epoch 03 | train_loss=0.242476 | val_RMSE_all=0.679318
Epoch 04 | train_loss=0.237282 | val_RMSE_all=0.671962
Epoch 05 | train_loss=0.233306 | val_RMSE_all=0.677236
Epoch 06 | train_loss=0.229964 | val_RMSE_all=0.675911
Epoch 07 | train_loss=0.226830 | val_RMSE_all=0.679792
Epoch 08 | train_loss=0.223834 | val_RMSE_all=0.681571
Epoch 09 | train_loss=0.220706 |

100%|██████████| 43597/43597 [10:01<00:00, 72.54it/s]


RuntimeError: No PTB windows created. Check PTBXL_DIR or ECG channel selection or R-peak extraction.